# Especies por clase en España y Portugal (GBIF)

Este cuaderno se apoya en el pipeline multi-especie de [`gbif_multiespecie_es_pt.ipynb`](./gbif_multiespecie_es_pt.ipynb): mismos países (`ES`, `PT`), mismos años y la misma carpeta de salida. No descarga el lote de ocurrencias ni pinta el mapa comparativo.

Sirve para **ver las especies** antes de elegir cuáles bajar. Un ranking único de `/v1/occurrence/search` se llena de aves. Aquí se piden cinco especies con coordenadas de cada clase, aunque tengan menos registros:

- Mammalia
- Squamata (lagartos y serpientes; en GBIF no salen como clase Reptilia)
- Amphibia
- Insecta
- Magnoliopsida y Liliopsida
- Aves

La misma lista la muestra la interfaz, en el botón **Especies con coordenadas**.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

from ingesta_gbif_sqlite import (
    CLASES_A_MOSTRAR,
    ESPECIES_POR_CLASE,
    listar_especies_con_coordenadas,
)

PAISES = ["ES", "PT"]
YEAR_RANGE = "2000,2026"
CARPETA_SALIDA = RAIZ / "salidas_gbif_multiespecie"
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

print("Clases:", ", ".join(CLASES_A_MOSTRAR))
print(f"Especies por clase: {ESPECIES_POR_CLASE}")
print(f"Salida: {CARPETA_SALIDA}")

## 1. Catálogo

Endpoints:

- `GET /v1/species/match?rank=CLASS` resuelve la clave de cada clase.
- `GET /v1/occurrence/search?classKey={clave}&facet=speciesKey&hasCoordinate=true` devuelve las especies con coordenadas de esa clase.
- `GET /v1/species/{speciesKey}` aporta el nombre, la clase y la familia.

In [ ]:
filas = listar_especies_con_coordenadas(
    paises=PAISES,
    year_range=YEAR_RANGE,
    limite=ESPECIES_POR_CLASE,
    on_progreso=print,
)
df_especies = pd.DataFrame(filas)
df_especies

## 2. Ver las clases y las familias

Cada bloque es una clase. La familia permite distinguir, dentro de los mamíferos, cánidos, suidos o cérvidos, y dentro de las aves, el grupo de cada especie.

In [ ]:
resumen_clases = (
    df_especies.groupby("clase", sort=False)
    .agg(
        especies=("nombre_cientifico", "count"),
        ocurrencias_con_coordenadas=("ocurrencias_con_coordenadas", "sum"),
    )
    .reset_index()
)
display(resumen_clases)

for clase, bloque in df_especies.groupby("clase", sort=False):
    print(f"\n{clase}")
    display(
        bloque[
            [
                "familia",
                "nombre_cientifico",
                "taxon_key",
                "ocurrencias_con_coordenadas",
            ]
        ].reset_index(drop=True)
    )

## 3. Guardar el catálogo y elegir especies para el cuaderno multi-especie

El CSV queda en `salidas_gbif_multiespecie/especies_por_clase_es_pt.csv`. La lista de nombres se puede copiar en `ESPECIES_OBJETIVO` de `gbif_multiespecie_es_pt.ipynb` para descargar ocurrencias y mapas. Conviene no copiar las 35 a la vez: ese cuaderno baja hasta 600 registros por país y por especie.

In [ ]:
ruta_csv = CARPETA_SALIDA / "especies_por_clase_es_pt.csv"
df_especies.to_csv(ruta_csv, index=False, encoding="utf-8-sig")
print(ruta_csv)

print("\nESPECIES_OBJETIVO = [")
for nombre in df_especies["nombre_cientifico"]:
    print(f'    "{nombre}",')
print("]")